# Activation Decay and Localized Amplitude Entropy

**Companion notebook to:** *A Content-Addressed Adaptive Knowledge Substrate for Distributed Epistemic Coordination*, §11.3  
**Author:** N. Joven · March 2026  

---

This notebook demonstrates two coupled scoring primitives implemented in Ket's traversal engine:

1. **Activation decay** — continuous exponential decay applied on read, parameterized by per-node half-life
2. **Localized amplitude entropy** — normalized Shannon entropy of a node's neighbor activation distribution; high entropy signals a structurally contested position

The two compose: entropy is computed over *decay-adjusted* activations. A node whose neighbors have all decayed to near-floor reads as low-entropy (quiescent) not high-entropy (contested).

**Graph fixture:** 11-node directed epistemic graph with three semantic clusters (Physics, Math, Philosophy) and two bridge nodes at structural holes. Bridge nodes are the expected sites of high entropy when multiple clusters are simultaneously active.

In [ ]:
import math
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

np.random.seed(42)
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',
    'grid.color': '#21262d',
    'grid.alpha': 0.8,
    'lines.linewidth': 1.5,
    'font.size': 10,
})

HEATMAP = LinearSegmentedColormap.from_list(
    'epistemic', ['#0d1117', '#1f4068', '#1b6ca8', '#e2a80a', '#f7d060']
)

CLUSTER_COLORS = {
    'physics':    '#3b82f6',
    'math':       '#22c55e',
    'philosophy': '#a855f7',
    'bridge':     '#f59e0b',
}

print('Setup complete.')

## §1 — Epistemic Graph Setup

Eleven nodes across four clusters. Bridge nodes (9, 10) sit at structural holes — positions where paths from multiple clusters converge. Signed edges encode epistemic relationships:
- **+1** (solid green): supports / implies  
- **−1** (dashed red): contradicts

In [ ]:
def build_epistemic_graph():
    G = nx.DiGraph()

    # (id, type, cluster, label, stored_activation, half_life_hours)
    node_data = [
        (0,  'fact',        'physics',    'Wave-particle\nduality',    1.00,  6.0),
        (1,  'fact',        'physics',    'Superposition\nprinciple',  0.90,  8.0),
        (2,  'hypothesis',  'physics',    'Wavefunction\ncollapse',    0.80,  4.0),
        (3,  'fact',        'math',       'Hilbert space',             0.85, 12.0),
        (4,  'fact',        'math',       'Spectral\ndecomp.',         0.75, 10.0),
        (5,  'derived',     'math',       'Linear\noperators',         0.70,  8.0),
        (6,  'observation', 'philosophy', 'Observer\neffect',          0.60,  3.0),
        (7,  'hypothesis',  'philosophy', 'Copenhagen\ninterp.',       0.65,  5.0),
        (8,  'hypothesis',  'philosophy', 'Many-worlds\ninterp.',      0.55,  5.0),
        (9,  'derived',     'bridge',     'Quantum\nformalism',        0.90, 20.0),
        (10, 'derived',     'bridge',     'Interpretation\ndispute',   0.50,  2.0),
    ]

    for nid, ntype, cluster, label, activation, half_life in node_data:
        G.add_node(nid, node_type=ntype, cluster=cluster, label=label,
                   activation=activation, half_life=half_life)

    # (from, to, weight, sign)
    edge_data = [
        (0,  1,  0.90,  1),
        (1,  2,  0.70,  1),
        (3,  4,  0.80,  1),
        (4,  5,  0.90,  1),
        (5,  9,  0.80,  1),
        (1,  9,  0.90,  1),
        (3,  9,  0.85,  1),
        (2,  10, 0.70,  1),
        (6,  10, 0.80,  1),
        (7,  10, 0.60, -1),
        (8,  10, 0.60, -1),
        (7,  8,  0.90, -1),
        (9,  7,  0.70,  1),
        (9,  8,  0.70,  1),
        (6,  7,  0.75,  1),
    ]

    for u, v, w, s in edge_data:
        G.add_edge(u, v, weight=w, sign=s)

    return G


G = build_epistemic_graph()
nodes_list = list(G.nodes())
n = len(nodes_list)

pos = {
    0: (-2.5,  1.5), 1: (-2.5,  0.0), 2: (-2.5, -1.5),
    3: ( 0.0,  1.5), 4: ( 0.0,  0.0), 5: ( 0.0, -1.0),
    6: ( 2.5,  1.5), 7: ( 2.5,  0.0), 8: ( 2.5, -1.5),
    9: (-1.2, -0.5), 10: (1.3, -0.5),
}

print(f'Graph: {n} nodes, {G.number_of_edges()} edges')
print()
print(f'{"Node":>4}  {"Cluster":>12}  {"Label":>26}  {"A₀":>6}  {"T₁/₂ (h)":>9}')
print('-' * 65)
for v in nodes_list:
    d = G.nodes[v]
    label = d['label'].replace('\n', ' ')
    print(f'{v:>4}  {d["cluster"]:>12}  {label:>26}  {d["activation"]:>6.2f}  {d["half_life"]:>9.1f}')

# Visualize base graph
fig, ax = plt.subplots(figsize=(12, 7))

node_colors = [CLUSTER_COLORS[G.nodes[v]['cluster']] for v in nodes_list]
node_sizes  = [G.nodes[v]['activation'] * 800 + 200 for v in nodes_list]
solid_edges  = [(u, v) for u, v, d in G.edges(data=True) if d['sign'] ==  1]
dashed_edges = [(u, v) for u, v, d in G.edges(data=True) if d['sign'] == -1]

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9, ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=solid_edges,  edge_color='#4ade80',
                       arrows=True, arrowsize=15, width=1.5,
                       connectionstyle='arc3,rad=0.1', ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=dashed_edges, edge_color='#f87171',
                       arrows=True, arrowsize=15, width=1.5, style='dashed',
                       connectionstyle='arc3,rad=0.1', ax=ax)
nx.draw_networkx_labels(G, pos, {v: G.nodes[v]['label'] for v in nodes_list},
                        font_size=7, font_color='white', ax=ax)

legend_elements = [
    Patch(facecolor='#3b82f6', label='Physics'),
    Patch(facecolor='#22c55e', label='Math'),
    Patch(facecolor='#a855f7', label='Philosophy'),
    Patch(facecolor='#f59e0b', label='Bridge (structural hole)'),
    Line2D([0],[0], color='#4ade80', label='Supports (+1)'),
    Line2D([0],[0], color='#f87171', linestyle='dashed', label='Contradicts (−1)'),
]
ax.legend(handles=legend_elements, loc='upper right', framealpha=0.3, fontsize=8)
ax.set_title('Epistemic Graph: Typed Nodes, Signed Edges, Structural Holes', pad=12)
ax.axis('off')
plt.tight_layout()
plt.savefig('graph_base.png', dpi=150, bbox_inches='tight')
plt.show()
print('Bridge nodes 9 and 10 sit at structural holes receiving paths from multiple clusters.')

## §2 — Decay Dynamics

Activation decays exponentially at read time:

$$A_{\text{decay}}(t) = A_0 \cdot 2^{-t / T_{1/2}}$$

floored at `activation_floor = 0.01`. The stored $A_0$ is never modified.

Different half-lives produce different decay rates. Bridge node 10 (`T₁/₂ = 2h`) decays quickly; bridge node 9 (`T₁/₂ = 20h`) persists. Cluster-interior nodes vary between 3h and 12h.

In [ ]:
ACTIVATION_FLOOR = 0.01


def decayed_activation(stored: float, elapsed_hours: float, half_life: float,
                        floor: float = ACTIVATION_FLOOR) -> float:
    """Exponential decay; floored to prevent reaching zero."""
    if half_life <= 0:
        return stored
    decay_factor = math.exp(-math.log(2) * elapsed_hours / half_life)
    return max(stored * decay_factor, floor)


def decay_all(activations: np.ndarray, half_lives: np.ndarray,
               elapsed_hours: float) -> np.ndarray:
    """Vectorized decay for all nodes."""
    decay_factors = np.exp(-np.log(2) * elapsed_hours / half_lives)
    return np.maximum(activations * decay_factors, ACTIVATION_FLOOR)


stored = np.array([G.nodes[v]['activation'] for v in nodes_list])
half_lives = np.array([G.nodes[v]['half_life']  for v in nodes_list])

# Activation table at key timepoints
timepoints = [0, 1, 4, 12, 24]
print(f'{"Node":<5} {"Label":<28} {"T₁/₂":>6}', end='')
for t in timepoints:
    print(f'  {f"t={t}h":>7}', end='')
print()
print('-' * (5 + 28 + 6 + len(timepoints) * 9 + 4))

for i, v in enumerate(nodes_list):
    label = G.nodes[v]['label'].replace('\n', ' ')
    print(f'{v:<5} {label:<28} {half_lives[i]:>6.1f}', end='')
    for t in timepoints:
        a = decayed_activation(stored[i], t, half_lives[i])
        print(f'  {a:>7.4f}', end='')
    print()

# Decay curves
t_range = np.linspace(0, 30, 300)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for i, v in enumerate(nodes_list):
    cluster = G.nodes[v]['cluster']
    color = CLUSTER_COLORS[cluster]
    lw = 2.5 if cluster == 'bridge' else 1.0
    alpha = 0.9 if cluster == 'bridge' else 0.5
    label = G.nodes[v]['label'].replace('\n', ' ') if cluster == 'bridge' else None
    curve = [decayed_activation(stored[i], t, half_lives[i]) for t in t_range]
    ax.plot(t_range, curve, color=color, lw=lw, alpha=alpha, label=label)

ax.axhline(ACTIVATION_FLOOR, color='#6b7280', lw=1, linestyle=':', label=f'Floor ({ACTIVATION_FLOOR})')
legend_elements = [
    Patch(facecolor='#3b82f6', label='Physics'),
    Patch(facecolor='#22c55e', label='Math'),
    Patch(facecolor='#a855f7', label='Philosophy'),
    Patch(facecolor='#f59e0b', label='Bridge'),
    Line2D([0],[0], color='#6b7280', linestyle=':', label='Activation floor'),
]
ax.legend(handles=legend_elements, fontsize=7)
ax.set_title('Activation Decay Curves Per Node (colored by cluster)')
ax.set_xlabel('Elapsed time (hours)')
ax.set_ylabel('Decay-adjusted activation')
ax.grid(True, alpha=0.3)

# Decay fraction remaining at t=4h and t=12h
ax = axes[1]
x = np.arange(n)
frac_4h  = np.array([decayed_activation(stored[i], 4,  half_lives[i]) / stored[i] for i in range(n)])
frac_12h = np.array([decayed_activation(stored[i], 12, half_lives[i]) / stored[i] for i in range(n)])
colors_by_cluster = [CLUSTER_COLORS[G.nodes[v]['cluster']] for v in nodes_list]

ax.bar(x - 0.2, frac_4h,  0.38, color=colors_by_cluster, alpha=0.85, label='Fraction at t=4h')
ax.bar(x + 0.2, frac_12h, 0.38, color=colors_by_cluster, alpha=0.45, label='Fraction at t=12h')
short_labels = [G.nodes[v]['label'].replace('\n', ' ')[:14] for v in nodes_list]
ax.set_xticks(x)
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=7)
ax.set_title('Decay Fraction Remaining at t=4h (dark) and t=12h (faded)')
ax.set_ylabel('A(t) / A₀')
ax.axhline(1.0, color='#6b7280', lw=0.8, linestyle=':')
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('decay_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nNode 10 (Interpretation dispute, T₁/₂=2h) decays most aggressively.')
print('Node 9 (Quantum formalism, T₁/₂=20h) persists longest.')

## §3 — Localized Amplitude Entropy

For node $v$ with neighbors $N(v)$, let $a_i$ be the decay-adjusted activation of neighbor $i$. Localized amplitude entropy is:

$$p_i = \frac{a_i}{\sum_j a_j} \qquad H(v) = -\sum_i p_i \ln p_i \qquad H_{\text{norm}}(v) = \frac{H(v)}{\ln |N(v)|}$$

$H_{\text{norm}} \in [0, 1]$. Nodes with fewer than 2 neighbors receive $H_{\text{norm}} = 0$.

- $H_{\text{norm}} \approx 0$: activation concentrated on one or few neighbors — clear dominant traversal direction  
- $H_{\text{norm}} \approx 1$: activation spread uniformly — no dominant direction, structurally contested

In [ ]:
def localized_amplitude_entropy(neighbor_activations: list[float]) -> float:
    """Normalized Shannon entropy of neighbor activation distribution.
    Returns H_norm in [0, 1]; 0.0 if fewer than 2 neighbors.
    """
    n = len(neighbor_activations)
    if n < 2:
        return 0.0
    total = sum(neighbor_activations)
    if total < 1e-12:
        return 0.0
    entropy = sum(
        -(a / total) * math.log(a / total)
        for a in neighbor_activations
        if a > 1e-12
    )
    max_entropy = math.log(n)
    if max_entropy < 1e-12:
        return 0.0
    return max(0.0, min(1.0, entropy / max_entropy))


def verdict(h_norm: float) -> str:
    if h_norm >= 0.75:
        return 'contested'
    elif h_norm >= 0.35:
        return 'mixed'
    else:
        return 'concentrated'


def compute_entropy_for_node(G, node, activations_map: dict) -> float:
    """Compute localized amplitude entropy at node using given activation map.
    Neighbors = predecessors (nodes that point INTO this node).
    """
    neighbors = list(G.predecessors(node))
    if len(neighbors) < 2:
        return 0.0
    neighbor_acts = [activations_map[nb] for nb in neighbors]
    return localized_amplitude_entropy(neighbor_acts)


# Compute at t=0 (no decay)
acts_t0 = {v: G.nodes[v]['activation'] for v in nodes_list}

print(f'{'Node':>4}  {'Label':>26}  {'Cluster':>12}  {'In-degree':>9}  {'H_norm':>8}  {'Verdict'}')
print('-' * 78)

entropies_t0 = {}
for v in nodes_list:
    h = compute_entropy_for_node(G, v, acts_t0)
    entropies_t0[v] = h
    label = G.nodes[v]['label'].replace('\n', ' ')
    cluster = G.nodes[v]['cluster']
    indegree = G.in_degree(v)
    print(f'{v:>4}  {label:>26}  {cluster:>12}  {indegree:>9}  {h:>8.4f}  {verdict(h)}')

In [ ]:
# Visualize entropy on the graph
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

h_values = np.array([entropies_t0[v] for v in nodes_list])
norm = Normalize(vmin=0, vmax=1)

ax = axes[0]
node_colors_entropy = [HEATMAP(norm(h)) for h in h_values]
node_sizes_act = [G.nodes[v]['activation'] * 800 + 200 for v in nodes_list]

nx.draw_networkx_nodes(G, pos, node_color=node_colors_entropy,
                       node_size=node_sizes_act, alpha=0.95, ax=ax)
nx.draw_networkx_edges(G, pos,
                       edgelist=[(u,v) for u,v,d in G.edges(data=True) if d['sign']==1],
                       edge_color='#4ade80', arrows=True, arrowsize=12, width=1.2,
                       connectionstyle='arc3,rad=0.1', ax=ax, alpha=0.5)
nx.draw_networkx_edges(G, pos,
                       edgelist=[(u,v) for u,v,d in G.edges(data=True) if d['sign']==-1],
                       edge_color='#f87171', arrows=True, arrowsize=12, width=1.2,
                       style='dashed', connectionstyle='arc3,rad=0.1', ax=ax, alpha=0.5)

# Overlay H_norm values
for v in nodes_list:
    x_v, y_v = pos[v]
    h = entropies_t0[v]
    ax.text(x_v, y_v - 0.32, f'H={h:.2f}', ha='center', fontsize=6.5,
            color='#c9d1d9', fontweight='bold' if h >= 0.75 else 'normal')

ax.set_title('Localized Amplitude Entropy at t=0\n(node color = H_norm, size = stored activation)',
             fontsize=9)
ax.axis('off')

# Bar chart of entropy by node
ax = axes[1]
bar_colors = [
    '#f87171' if h >= 0.75 else
    '#fbbf24' if h >= 0.35 else
    '#4ade80'
    for h in h_values
]
x = np.arange(n)
ax.bar(x, h_values, color=bar_colors, alpha=0.85, edgecolor='#30363d', linewidth=0.5)
ax.axhline(0.75, color='#f87171', lw=1.2, linestyle='--', label='Contested threshold (0.75)')
ax.axhline(0.35, color='#fbbf24', lw=1.0, linestyle=':', label='Mixed threshold (0.35)')
short_labels = [G.nodes[v]['label'].replace('\n', ' ')[:14] for v in nodes_list]
ax.set_xticks(x)
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('H_norm')
ax.set_title('Localized Amplitude Entropy at t=0\n(red = contested, amber = mixed, green = concentrated)')
ax.legend(fontsize=8)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('entropy_t0.png', dpi=150, bbox_inches='tight')
plt.show()

contested = [v for v in nodes_list if entropies_t0[v] >= 0.75]
print(f'\nContested nodes (H_norm >= 0.75): {[G.nodes[v]["label"].replace(chr(10)," ") for v in contested]}')
print('These are the nodes receiving balanced activation from multiple directions.')

## §4 — Decay + Entropy Composition

Entropy is computed over **decay-adjusted** activations. This matters: a node with many neighbors that have all decayed to near-floor reads as low-entropy (quiescent, not contested).

**Scenario A** — quiescence: all neighbors of node 9 (Quantum formalism) decay. H_norm should fall toward 0 as the distribution collapses to the activation floor.

**Scenario B** — persistent contest: node 10 (Interpretation dispute) has neighbors decaying at different rates. One cluster remains active, the other decays. H_norm tracks the imbalance.

In [ ]:
t_hours = np.linspace(0, 30, 150)

def entropy_timeseries(G, target_node, stored_acts, half_lives_map, t_range):
    """Entropy at target_node over a range of elapsed times."""
    result = []
    predecessors = list(G.predecessors(target_node))
    for t in t_range:
        decayed = {
            nb: decayed_activation(stored_acts[nb], t, half_lives_map[nb])
            for nb in predecessors
        }
        h = localized_amplitude_entropy(list(decayed.values()))
        result.append(h)
    return np.array(result)


stored_acts = {v: G.nodes[v]['activation'] for v in nodes_list}
half_lives_map = {v: G.nodes[v]['half_life'] for v in nodes_list}

# Bridge node 9 predecessors: nodes 1, 3, 5 (all from Physics/Math clusters)
h9 = entropy_timeseries(G, 9, stored_acts, half_lives_map, t_hours)

# Bridge node 10 predecessors: nodes 2, 6, 7, 8 (Physics + Philosophy clusters)
h10 = entropy_timeseries(G, 10, stored_acts, half_lives_map, t_hours)

# Scenario A: freeze node 9's predecessor activations at floor (simulating all decayed)
frozen_acts = dict(stored_acts)
for nb in G.predecessors(9):
    frozen_acts[nb] = ACTIVATION_FLOOR
frozen_half_lives = {v: half_lives_map[v] for v in nodes_list}
# At this point all predecessors of 9 are at floor: uniform distribution → high entropy
# But since all are equal, the distribution IS uniform → H_norm = 1.0
# The key is that *all* are near floor — no dominant one
# Distinguish quiescence: all near floor vs. one active among many near floor

# Scenario A: artificially set one predecessor of node 9 to high activation
dominant_acts = dict(stored_acts)
preds_9 = list(G.predecessors(9))
dominant_acts[preds_9[0]] = 1.0  # node 1 stays fully active
for nb in preds_9[1:]:
    dominant_acts[nb] = ACTIVATION_FLOOR  # others decay to floor

h9_dominant = entropy_timeseries(G, 9, dominant_acts,
                                  {v: 1e9 for v in nodes_list},  # no decay — static
                                  t_hours)

# Scenario B: fast-decaying node 10 neighbors vs slow
# Predecessors of node 10: 2 (T₁/₂=4h), 6 (T₁/₂=3h), 7 (T₁/₂=5h), 8 (T₁/₂=5h)
# All decay rapidly — show entropy falling
h10_natural = entropy_timeseries(G, 10, stored_acts, half_lives_map, t_hours)

# Variant: keep nodes 7 and 8 (philosophy cluster) alive, let 2 and 6 decay
persistent_acts = dict(stored_acts)
persistent_half_lives = dict(half_lives_map)
for nb in G.predecessors(10):
    if G.nodes[nb]['cluster'] == 'philosophy':
        persistent_half_lives[nb] = 50.0  # very slow decay for philosophy nodes

h10_persistent = entropy_timeseries(G, 10, stored_acts, persistent_half_lives, t_hours)


fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Decay + Entropy Composition', fontsize=12)

# Panel 1: entropy at bridge node 9 over time
ax = axes[0]
ax.plot(t_hours, h9, color='#f59e0b', lw=2, label='Node 9: natural decay')
ax.axhline(0.75, color='#f87171', lw=1, linestyle='--', alpha=0.6, label='Contested threshold')
ax.set_title('Node 9 (Quantum Formalism)\nBridge: Physics + Math inputs')
ax.set_xlabel('Elapsed time (hours)')
ax.set_ylabel('H_norm')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# Panel 2: entropy at node 9 with dominant vs. uniform predecessor activations
ax = axes[1]
# Uniform floor activations
h9_allfloor = localized_amplitude_entropy([ACTIVATION_FLOOR] * len(preds_9))
ax.axhline(h9_allfloor, color='#6b7280', lw=1.5, linestyle=':', label=f'All at floor (H={h9_allfloor:.2f})')
ax.plot([0], [h9_dominant[0]], 'o', color='#60a5fa', ms=8,
        label=f'One dominant, others at floor (H={h9_dominant[0]:.2f})')

preds_9_acts = [dominant_acts[nb] for nb in preds_9]
ax.barh(range(len(preds_9)), preds_9_acts,
        color=['#f59e0b' if nb == preds_9[0] else '#4ade80' for nb in preds_9],
        alpha=0.7)
ax.set_yticks(range(len(preds_9)))
ax.set_yticklabels([G.nodes[nb]['label'].replace('\n', ' ') for nb in preds_9], fontsize=8)
ax.set_xlabel('Activation')
ax.set_title(f'Quiescent vs. Contested at Node 9\nKey: all-at-floor ≠ one-dominant')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3, axis='x')

# Panel 3: entropy at node 10 — natural vs persistent philosophy cluster
ax = axes[2]
ax.plot(t_hours, h10_natural,   color='#f59e0b', lw=2,   label='Node 10: natural decay (all)')
ax.plot(t_hours, h10_persistent, color='#a855f7', lw=2, linestyle='--',
        label='Node 10: philosophy cluster persists')
ax.axhline(0.75, color='#f87171', lw=1, linestyle='--', alpha=0.6, label='Contested threshold')
ax.set_title('Node 10 (Interpretation Dispute)\nContested bridge — fast-decaying predecessors')
ax.set_xlabel('Elapsed time (hours)')
ax.set_ylabel('H_norm')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('entropy_decay_composition.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key observation:')
print(f'  All predecessors of node 9 at floor → H_norm = {h9_allfloor:.3f} (uniform but quiescent)')
print(f'  One dominant, others at floor      → H_norm = {h9_dominant[0]:.3f} (concentrated = not contested)')
print('  Quiescent (all near floor) ≠ contested (many active, roughly equal).')
print('  The entropy signal is grounded in active activation, not just neighbor count.')

## §5 — Traversal Priority

The `NodeScore.traversal_priority()` formula from the Ket spec:

```
base = 0.3×correctness + 0.2×efficiency + 0.1×style + 0.2×completeness + 0.2×decay_adjusted_activation

entropy_modifier = 1.5  if amplitude_entropy > 0.75  (contested)
                 = 1.1  if amplitude_entropy > 0.50  (mixed)
                 = 1.0  otherwise                    (concentrated)

traversal_priority = base × entropy_modifier
```

The modifier raises investigation priority for contested nodes. We rank all 11 nodes at t=0 and t=4h.

In [ ]:
def traversal_priority(correctness: float, efficiency: float, style: float,
                        completeness: float, decay_adjusted_activation: float,
                        amplitude_entropy: float) -> float:
    base = (0.3 * correctness
          + 0.2 * efficiency
          + 0.1 * style
          + 0.2 * completeness
          + 0.2 * decay_adjusted_activation)
    if amplitude_entropy > 0.75:
        modifier = 1.5
    elif amplitude_entropy > 0.50:
        modifier = 1.1
    else:
        modifier = 1.0
    return base * modifier


# Use stored scoring dimensions (simplified: use activation as proxy for all dimensions)
def node_score(G, v, elapsed_hours):
    a0 = G.nodes[v]['activation']
    hl = G.nodes[v]['half_life']
    a_dec = decayed_activation(a0, elapsed_hours, hl)

    acts_at_t = {
        u: decayed_activation(G.nodes[u]['activation'], elapsed_hours, G.nodes[u]['half_life'])
        for u in nodes_list
    }
    h = compute_entropy_for_node(G, v, acts_at_t)

    # Use activation as a stand-in for all four quality dimensions
    tp = traversal_priority(
        correctness=a0, efficiency=a0, style=a0, completeness=a0,
        decay_adjusted_activation=a_dec,
        amplitude_entropy=h
    )
    return a_dec, h, tp


for elapsed in [0, 4]:
    print(f'\n--- Traversal priority at t={elapsed}h ---')
    print(f'{"Rank":>4}  {"Node":>4}  {"Label":>26}  {"A(t)":>8}  {"H_norm":>8}  {"Verdict":>12}  {"Priority":>9}')
    print('-' * 80)

    scores = []
    for v in nodes_list:
        a_dec, h, tp = node_score(G, v, elapsed)
        scores.append((v, a_dec, h, tp))

    scores.sort(key=lambda x: -x[3])
    for rank, (v, a_dec, h, tp) in enumerate(scores, 1):
        label = G.nodes[v]['label'].replace('\n', ' ')
        print(f'{rank:>4}  {v:>4}  {label:>26}  {a_dec:>8.4f}  {h:>8.4f}  {verdict(h):>12}  {tp:>9.4f}')

In [ ]:
# Visualize priority shift over time
t_snap = [0, 2, 4, 8, 12, 24]
priority_matrix = np.zeros((n, len(t_snap)))

for j, t in enumerate(t_snap):
    for i, v in enumerate(nodes_list):
        _, _, tp = node_score(G, v, t)
        priority_matrix[i, j] = tp

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(priority_matrix, aspect='auto', cmap='plasma', interpolation='nearest')
ax.set_xticks(range(len(t_snap)))
ax.set_xticklabels([f't={t}h' for t in t_snap])
ax.set_yticks(range(n))
ax.set_yticklabels([G.nodes[v]['label'].replace('\n', ' ') + f' (cluster: {G.nodes[v]["cluster"]})'
                    for v in nodes_list], fontsize=8)
plt.colorbar(im, ax=ax, label='Traversal priority')
ax.set_title('Traversal Priority Over Time (decay + entropy modifier combined)', pad=10)
plt.tight_layout()
plt.savefig('traversal_priority_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Contested nodes start with elevated priority (entropy modifier 1.5×).')
print('As predecessors decay, entropy falls and the modifier drops — but so does activation.')
print('The combined signal: high priority = either recently active OR structurally contested.')

## §6 — Contradiction Detection

Node 10 ("Interpretation dispute") has four predecessors:
- Node 2 (Wavefunction collapse, Physics) — supports
- Node 6 (Observer effect, Philosophy) — supports  
- Node 7 (Copenhagen, Philosophy) — contradicts  
- Node 8 (Many-worlds, Philosophy) — contradicts

**Baseline:** Only one interpretation active (node 7 or 8 alone).  
**Experiment:** Both interpretations simultaneously active (nodes 7 and 8 both at high activation).  

Prediction: co-activation of mutually contradictory nodes 7 and 8 raises H_norm at node 10 → contested verdict.

In [ ]:
def entropy_check(G, target_node, activation_override: dict | None = None) -> dict:
    """Compute entropy_check result for a node, optionally overriding activations."""
    acts = {v: G.nodes[v]['activation'] for v in G.nodes()}
    if activation_override:
        acts.update(activation_override)

    predecessors = list(G.predecessors(target_node))
    neighbor_acts = {nb: acts[nb] for nb in predecessors}
    total = sum(neighbor_acts.values())
    weights = {nb: a / total for nb, a in neighbor_acts.items()} if total > 1e-12 else {}
    h = localized_amplitude_entropy(list(neighbor_acts.values()))

    return {
        'node': target_node,
        'label': G.nodes[target_node]['label'].replace('\n', ' '),
        'amplitude_entropy': h,
        'neighbor_count': len(predecessors),
        'neighbor_distribution': [
            {
                'node': nb,
                'label': G.nodes[nb]['label'].replace('\n', ' '),
                'activation': neighbor_acts[nb],
                'weight': weights.get(nb, 0.0),
            }
            for nb in predecessors
        ],
        'verdict': verdict(h),
    }


HIGH = 0.95
LOW  = 0.05

scenarios = {
    'baseline (Copenhagen only)':       {7: HIGH, 8: LOW},
    'baseline (Many-worlds only)':      {7: LOW,  8: HIGH},
    'both active (contradiction)':      {7: HIGH, 8: HIGH},
    'both suppressed':                  {7: LOW,  8: LOW},
}

results = {}
for name, override in scenarios.items():
    result = entropy_check(G, 10, activation_override=override)
    results[name] = result
    print(f'\nScenario: {name}')
    print(f'  amplitude_entropy = {result["amplitude_entropy"]:.4f}')
    print(f'  verdict           = {result["verdict"]}')
    print(f'  neighbor distribution:')
    for nb_info in result['neighbor_distribution']:
        print(f'    Node {nb_info["node"]} ({nb_info["label"]:>26}): '
              f'A={nb_info["activation"]:.3f}, weight={nb_info["weight"]:.3f}')

In [ ]:
# Sweep: entropy at node 10 as we vary node 7 activation (node 8 held fixed at HIGH)
sweep_vals = np.linspace(0.01, 0.99, 100)
h_sweep_node7 = []
h_sweep_node8 = []
h_sweep_both  = []

for val in sweep_vals:
    h7 = entropy_check(G, 10, {7: val,  8: LOW })['amplitude_entropy']
    h8 = entropy_check(G, 10, {7: LOW,  8: val })['amplitude_entropy']
    hb = entropy_check(G, 10, {7: val,  8: val })['amplitude_entropy']
    h_sweep_node7.append(h7)
    h_sweep_node8.append(h8)
    h_sweep_both.append(hb)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Contradiction Detection via Localized Amplitude Entropy at Node 10', fontsize=11)

ax = axes[0]
ax.plot(sweep_vals, h_sweep_node7, color='#3b82f6', lw=2, label='Vary node 7 (node 8 low)')
ax.plot(sweep_vals, h_sweep_node8, color='#a855f7', lw=2, label='Vary node 8 (node 7 low)')
ax.plot(sweep_vals, h_sweep_both,  color='#f87171', lw=2.5, linestyle='--',
        label='Vary both equally (contradiction)')
ax.axhline(0.75, color='#f87171', lw=1, linestyle=':', alpha=0.6, label='Contested threshold')
ax.set_xlabel('Activation value (sweep)')
ax.set_ylabel('H_norm at node 10')
ax.set_title('H_norm at "Interpretation Dispute" (node 10)\nvs. activation of interpretation nodes')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# Bar chart: entropy across all four scenarios
ax = axes[1]
scenario_names = list(results.keys())
scenario_entropies = [results[s]['amplitude_entropy'] for s in scenario_names]
scenario_colors = [
    '#f87171' if h >= 0.75 else
    '#fbbf24' if h >= 0.35 else
    '#4ade80'
    for h in scenario_entropies
]
y = np.arange(len(scenario_names))
ax.barh(y, scenario_entropies, color=scenario_colors, alpha=0.85, edgecolor='#30363d')
ax.axvline(0.75, color='#f87171', lw=1, linestyle='--', alpha=0.6)
ax.axvline(0.35, color='#fbbf24', lw=1, linestyle=':',  alpha=0.6)
ax.set_yticks(y)
ax.set_yticklabels(scenario_names, fontsize=8)
ax.set_xlabel('H_norm at node 10')
ax.set_title('Entropy Under Four Activation Scenarios\n(red = contested → Tier 2 investigation)')
for i, (h, verd) in enumerate(zip(scenario_entropies,
                                   [results[s]['verdict'] for s in scenario_names])):
    ax.text(h + 0.01, i, f'{h:.3f} ({verd})', va='center', fontsize=8)
ax.set_xlim(0, 1.15)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('contradiction_detection.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nConclusion:')
print('  Co-activating mutually contradictory interpretations (7 + 8) at node 10')
print('  produces a contested verdict via amplitude entropy.')
print('  One interpretation alone → mixed or concentrated.')
print('  Both suppressed → concentrated (quiescent).')
print('  The entropy signal cleanly distinguishes contested from merely active.')

## §7 — Contested Subgraph Resolution

When amplitude entropy identifies a contested node, the scorer faces the same situation as
degenerate perturbation theory: the standard method (score-based ranking) cannot break the
tie, because the tie exists *by design* — the graph has a local symmetry the scorer cannot see.

The correct response is to operate **within** the contested subspace using a method the
symmetry doesn't break. Here: extract the contested subgraph (contested nodes + their
predecessors) and run PageRank over it. The stationary distribution is determined by
local topology — not by the scorer's weights — so it returns resolution weights that
reflect the subgraph's own structure.

This closes the open calibration problem for the contested-node case: instead of arbitrary
tie-breaking, the system returns principled resolution weights derived from the same
content-addressed structure that detected the contest.

## Summary

This notebook demonstrates two scoring primitives implemented in Ket's traversal engine:

**Activation decay (§2):**  
- Exponential decay applied on read; stored activation unchanged (content-addressing invariant preserved)  
- Floored at 0.01 — quiescent nodes remain findable at minimal traversal depth  
- Per-node half-life: bridge nodes with short T₁/₂ lose relevance faster than durable cluster-interior facts

**Localized amplitude entropy (§3):**  
- Normalized Shannon entropy of decay-adjusted neighbor activation distribution  
- O(degree) per node, no global graph state required  
- Bridge nodes at structural holes consistently register as contested at t=0  
- Quiescence (all neighbors at floor) correctly produces low entropy — `contested` requires active neighbors, not just many neighbors  
- Co-activating contradictory nodes raises entropy at the bridge receiving both paths → Tier 2 investigation triggered

**Composition:** entropy over decay-adjusted activations means a formerly contested node that has gone quiescent no longer signals for investigation. The signal is time-sensitive by construction.

In [ ]:
# --- Visualize: contested subgraph highlighted, resolution weights as node size ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('§7: Contested Subgraph Extraction and PageRank Resolution', fontsize=11)

# Panel 1: full graph, contested subgraph highlighted
ax = axes[0]
all_pr_full = nx.pagerank(subgraph_t0, alpha=0.85) if len(subgraph_t0.nodes()) > 0 else {}

node_colors_full = []
node_sizes_full = []
for v in nodes_list:
    if v in contested_t0:
        node_colors_full.append('#f87171')       # red = contested
    elif v in subgraph_t0.nodes():
        node_colors_full.append('#fbbf24')       # amber = subgraph (predecessors)
    else:
        node_colors_full.append('#374151')       # grey = outside
    node_sizes_full.append(G.nodes[v]['activation'] * 600 + 150)

nx.draw_networkx_nodes(G, pos, node_color=node_colors_full,
                       node_size=node_sizes_full, alpha=0.9, ax=ax)
nx.draw_networkx_edges(G, pos,
                       edgelist=[(u,v) for u,v,d in G.edges(data=True) if d['sign']==1],
                       edge_color='#4ade80', arrows=True, arrowsize=12, width=1.0,
                       connectionstyle='arc3,rad=0.1', ax=ax, alpha=0.4)
nx.draw_networkx_edges(G, pos,
                       edgelist=[(u,v) for u,v,d in G.edges(data=True) if d['sign']==-1],
                       edge_color='#f87171', arrows=True, arrowsize=12, width=1.0,
                       style='dashed', connectionstyle='arc3,rad=0.1', ax=ax, alpha=0.4)
nx.draw_networkx_labels(G, pos, {v: G.nodes[v]['label'] for v in nodes_list},
                        font_size=6.5, font_color='white', ax=ax)
legend_elements = [
    Patch(facecolor='#f87171', label='Contested (H_norm ≥ 0.75)'),
    Patch(facecolor='#fbbf24', label='Subgraph (predecessors)'),
    Patch(facecolor='#374151', label='Outside subgraph'),
]
ax.legend(handles=legend_elements, loc='upper right', framealpha=0.3, fontsize=8)
ax.set_title('Contested Subgraph\n(red = contested, amber = predecessors)', fontsize=9)
ax.axis('off')

# Panel 2: resolution weights as bar chart
ax = axes[1]
if resolution_t0:
    res_nodes = sorted(resolution_t0.keys())
    res_weights = [resolution_t0[v] for v in res_nodes]
    res_labels = [G.nodes[v]['label'].replace('\n', ' ') for v in res_nodes]
    bar_colors = ['#f87171' if v in contested_t0 else '#fbbf24' for v in res_nodes]

    y = np.arange(len(res_nodes))
    ax.barh(y, res_weights, color=bar_colors, alpha=0.85, edgecolor='#30363d')
    ax.set_yticks(y)
    ax.set_yticklabels(res_labels, fontsize=9)
    ax.set_xlabel('Resolution weight (normalized PageRank)')
    ax.set_title('Resolution Weights Returned to Scorer\n(PageRank over contested subgraph)',
                 fontsize=9)
    for i, (v, w) in enumerate(zip(res_nodes, res_weights)):
        label_str = f'{w:.3f}' + (' ← contested' if v in contested_t0 else '')
        ax.text(w + 0.005, i, label_str, va='center', fontsize=8)
    ax.set_xlim(0, max(res_weights) * 1.35)
    ax.grid(True, alpha=0.3, axis='x')
else:
    ax.text(0.5, 0.5, 'No contested nodes at t=0', ha='center', va='center',
            transform=ax.transAxes)
    ax.axis('off')

plt.tight_layout()
plt.savefig('contested_subgraph_resolution.png', dpi=150, bbox_inches='tight')
plt.show()

# Show how resolution weights shift as the contradiction scenario activates
print("\n=== Resolution weights across activation scenarios ===")
print(f"{'Scenario':>35s}  {'H_norm(10)':>11s}  {'Verdict':>11s}  {'Resolution (node 10)':>20s}")
print("-" * 82)

for name, override in scenarios.items():
    # Temporarily override activations to simulate scenario
    G_temp = G.copy()
    for nid_k, val in override.items():
        G_temp.nodes[nid_k]['activation'] = val
    acts_temp = {v: G_temp.nodes[v]['activation'] for v in G_temp.nodes()}
    h10 = compute_entropy_for_node(G_temp, 10, acts_temp)
    sub_temp, cont_temp = extract_contested_subgraph(G_temp, elapsed_hours=0.0)
    if 10 in cont_temp:
        res_temp = resolve_by_walk(sub_temp, cont_temp)
        w10 = res_temp.get(10, 0.0)
        res_str = f"node 10: {w10:.3f}"
    else:
        res_str = "not contested"
    print(f"{name:>35s}  {h10:>11.4f}  {verdict(h10):>11s}  {res_str:>20s}")

In [ ]:
def extract_contested_subgraph(G, elapsed_hours=0.0, threshold=0.75):
    """
    Extract the contested subgraph: all nodes with H_norm >= threshold,
    plus their predecessors (the 'degenerate subspace' in PT terms).

    Returns (subgraph, contested_nodes).
    """
    acts = {
        v: decayed_activation(G.nodes[v]['activation'], elapsed_hours,
                               G.nodes[v]['half_life'])
        for v in G.nodes()
    }
    contested = [
        v for v in G.nodes()
        if compute_entropy_for_node(G, v, acts) >= threshold
    ]
    if not contested:
        return G.subgraph([]).copy(), []

    subgraph_nodes = set(contested)
    for v in contested:
        subgraph_nodes.update(G.predecessors(v))

    return G.subgraph(subgraph_nodes).copy(), contested


def resolve_by_walk(subgraph, contested_nodes, alpha=0.85):
    """
    Run PageRank over the contested subgraph and return resolution weights
    for the contested nodes, normalized to sum to 1.

    PageRank is the 'diagonalize within the subspace' step: it uses the
    subgraph's own topology to break ties, not the scorer's external weights.
    """
    if len(subgraph.nodes()) == 0 or not contested_nodes:
        k = max(len(contested_nodes), 1)
        return {v: 1.0 / k for v in contested_nodes}

    pr = nx.pagerank(subgraph, alpha=alpha)
    total = sum(pr.get(v, 0.0) for v in contested_nodes)
    if total < 1e-12:
        k = len(contested_nodes)
        return {v: 1.0 / k for v in contested_nodes}

    return {v: pr.get(v, 0.0) / total for v in contested_nodes}


# --- Demonstration at t=0 ---
subgraph_t0, contested_t0 = extract_contested_subgraph(G, elapsed_hours=0.0)
resolution_t0 = resolve_by_walk(subgraph_t0, contested_t0)

print("=== Contested Subgraph at t=0 ===")
print(f"Contested nodes (H_norm >= 0.75): {[G.nodes[v]['label'].replace(chr(10),' ') for v in contested_t0]}")
print(f"Subgraph nodes (contested + predecessors): {list(subgraph_t0.nodes())}")
print(f"Subgraph edges: {list(subgraph_t0.edges())}")
print()

print("Resolution weights (PageRank, α=0.85):")
print(f"{'Node':>5}  {'Label':>26}  {'Cluster':>12}  {'A₀':>6}  {'H_norm':>8}  {'PR weight':>10}")
print("-" * 75)

all_pr = nx.pagerank(subgraph_t0, alpha=0.85) if len(subgraph_t0.nodes()) > 0 else {}
acts_t0 = {v: G.nodes[v]['activation'] for v in G.nodes()}

for v in sorted(subgraph_t0.nodes()):
    label = G.nodes[v]['label'].replace('\n', ' ')
    cluster = G.nodes[v]['cluster']
    a0 = G.nodes[v]['activation']
    h = compute_entropy_for_node(G, v, acts_t0)
    pr_w = resolution_t0.get(v, all_pr.get(v, 0.0))
    is_contested = '← contested' if v in contested_t0 else ''
    print(f"{v:>5}  {label:>26}  {cluster:>12}  {a0:>6.2f}  {h:>8.4f}  {pr_w:>10.4f}  {is_contested}")

print()
print("Resolution weights are returned to the scorer in place of arbitrary tie-breaking.")
print("They reflect topology (predecessor structure, edge weights) — not scorer opinion.")